# Task 1 — Profile the Data

Establish exactly how messy `raw_pos_export.csv` is, with real counts, before assuming anything about its structure. Every finding here directly shapes the assumption tests in `02_validate_assumptions.ipynb` and the reconstruction rule in `03_reconstruct_invoices.ipynb`.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw_pos_export.csv")

# check: confirms the file loaded as expected before profiling anything
df.info()

## Null rates

Raw counts are hard to read at a glance -- convert to percentages.

In [ ]:
# null rate per column, as a percentage, easier to read than raw counts
null_rates = (df.isnull().mean() * 100).round(1)

# check: invoice_number and line_note should stand out as mostly empty
print(null_rates)

**Finding:** `invoice_number` is populated on only ~29% of rows -- not usable as a reliable grouping key on its own. `line_note` is populated on only ~5% of rows, which is expected (it's a free-text field only filled in for exceptions like returns or gift wrap).

## Duplicate rows

In [ ]:
# naive check across all columns, including row_id
naive_dupes = df.duplicated().sum()

# check: this comes back 0 -- but that's misleading, see next cell
print("duplicates including row_id:", naive_dupes)

In [ ]:
# row_id is a sequential export number, not a business key -- it makes
# every row look "unique" even when the real transaction data is identical.
# drop it before checking for true duplicates
real_dupes = df.drop(columns="row_id").duplicated().sum()

# check: this is the real duplicate count, likely a POS double-submit glitch
print("duplicates excluding row_id:", real_dupes)

**Finding:** 2 exact duplicate rows once `row_id` is excluded -- the naive check hid this completely.

## Distinct value counts

In [ ]:
# distinct value counts for the columns most likely to carry inconsistency
counts = df[["store_id", "category_raw", "payment_method"]].nunique()

# check: category_raw showing more values than expected is a red flag
print(counts)

In [ ]:
# does case-folding collapse category_raw down to fewer real categories?
case_folded_count = df["category_raw"].str.lower().nunique()

# check: if this is lower than the raw count above, it's a casing problem,
# not genuinely different categories
print("distinct categories after lowercasing:", case_folded_count)

**Finding:** `category_raw` shows 8 distinct values but collapses to 4 once case-folded (e.g. `Home` vs `HOME`) -- inconsistent casing, not real categories.

## Store reference reconciliation

Compare store IDs in the export against the reference table -- a preview of the migration-style reconciliation this project is meant to practice.

In [ ]:
# load the store reference table so we can compare it against the export
stores = pd.read_csv("../data/store_lookup.csv")

# set difference: store_ids present in the export but missing from the
# lookup table -- these are unmapped stores we can't get a name for
print("in export but not lookup:", set(df["store_id"]) - set(stores["store_id"]))

# the reverse: store_ids in the lookup table that never actually show up
# in the export -- these are reference entries with no matching activity
print("in lookup but not export:", set(stores["store_id"]) - set(df["store_id"]))

**Finding:** `ST99` appears in the export but not in the lookup table; `ST04` exists in the lookup table but is never used in the export.

## Summary

| Check | Result |
|---|---|
| Rows / columns | 131 rows, 17 columns |
| `invoice_number` completeness | ~29% populated -- not a reliable grouping key |
| `line_note` completeness | ~5% populated -- expected, exception-only field |
| Exact duplicates | 2, hidden by `row_id` unless explicitly excluded |
| `category_raw` | 8 raw values, 4 real categories once case-folded |
| Store reconciliation | `ST99` unmapped in export; `ST04` unused in lookup |

These findings carry forward directly: the `invoice_number` gap motivates Task 2's assumption testing and Task 3's reconstruction rule; the store mismatch feeds Task 5's reconciliation step.